# 15 · Simulation test: is the generator essentially the formula?

**Question:** on the real data, the 29-term formula (`SPEC4`) beats CatBoost by ~0.006, a 75/25 blend adds ~+0.0007, and CatBoost boosted from the formula adds ~+0.0002. Are those numbers what we'd see **if the formula were the true generator**, or does the real data contain structure the formula misses?

**Method:**
1. Treat the formula fitted on all labeled quotes as the **known truth** p*.
2. Draw simulated outcomes y* ~ Bernoulli(p*) for the same 29,496 quotes (same features; 5 independent simulations).
3. On each simulation, rerun the identical pipeline on the seed-0 folds: formula refit, CatBoost (`07` parameters, tree count chosen as in `cv_score`), the 0.75/0.25 blend, and residual CatBoost on the formula baseline. Also score the **oracle** (the true p*), which is the best AUC achievable on that simulation.
4. Compare each gap with the same quantity on the real data.

**How to read it:**
- If the real-data gaps fall inside the simulation range, the real data behaves exactly like "formula + coin flips".
- If CatBoost or the residual model does clearly **better on real data than in simulation**, the real generator has structure the formula lacks.

In [1]:
import json, sys
import numpy as np, pandas as pd
import catboost as cbm
from sklearn.metrics import roc_auc_score
sys.path.insert(0, '.')
import common as C, features as FE, formula as FM
import formula_steps as FS
L, y, lab = FS.L, FS.y, FS.lab
PC = json.load(open('params_cats.json')); FX = PC['features']
LC = FE.add_cats(FE.add_static(lab)); cats = [c for c in FX if not pd.api.types.is_numeric_dtype(LC[c])]
FOLDS0 = [f for f in C.folds(lab) if f[0] == 0]
RP = PC['params'] | {'learning_rate': 0.02}
sig = lambda z: 1 / (1 + np.exp(-z))
truth = FM.Formula(FS.SPEC4, 0.1, '2era').fit(L, y)
p_star = truth.predict(L)
print(f'truth: mean p* {p_star.mean():.4f} (real base rate {y.mean():.4f}); oracle AUC on the real labels {roc_auc_score(y, p_star):.4f} (in-sample)')

def pipeline(yy, tag):
    """Formula, CatBoost, blend, residual CatBoost on the seed-0 folds for outcome vector yy. Returns mean fold AUCs."""
    Lf, Lc = L.assign(CONVERT_IND=yy), LC.assign(CONVERT_IND=yy)
    fx = C.cv_score(f'{tag}_fx', Lf, FM.columns(FS.SPEC4), 'formula', {'spec': FS.SPEC4, 'C': 0.1, 'intercepts': '2era'}, seeds=(0,), log=False)
    cb = C.cv_score(f'{tag}_cb', Lc, FX, 'cb', PC['params'], seeds=(0,), log=False)
    P_f, P_c = fx['oof_all'][0], cb['oof_all'][0]
    out = {'formula': [], 'CatBoost': [], 'blend 0.75/0.25': [], 'formula + residual CB': [], 'oracle (true p*)': []}
    curves, base = [], []
    for s, k, tr, va in FOLDS0:
        m = FM.Formula(FS.SPEC4, 0.1, '2era').fit(Lf.iloc[tr], yy[tr])
        btr, bva = m.decision(Lf.iloc[tr]), m.decision(Lf.iloc[va])
        Xva = C.cb_X(Lc.iloc[va][FX], cats)
        r = cbm.CatBoostClassifier(**RP, iterations=2000, early_stopping_rounds=200, use_best_model=False)
        r.fit(cbm.Pool(C.cb_X(Lc.iloc[tr][FX], cats), yy[tr], cat_features=cats, baseline=btr), eval_set=cbm.Pool(Xva, yy[va], cat_features=cats, baseline=bva))
        curves.append(np.array(r.get_evals_result()['validation']['AUC'])); base.append(roc_auc_score(yy[va], bva))
        for n, p in (('formula', P_f[va]), ('CatBoost', P_c[va]), ('blend 0.75/0.25', 0.75 * P_f[va] + 0.25 * P_c[va]), ('oracle (true p*)', p_star[va])):
            out[n].append(roc_auc_score(yy[va], p))
    Lm = max(map(len, curves)); bi = int(np.mean([np.pad(c, (0, Lm - len(c)), mode='edge') for c in curves], axis=0).argmax())
    out['formula + residual CB'] = [c[min(bi, len(c) - 1)] for c in curves]
    res = {k: float(np.mean(v)) for k, v in out.items()}
    res |= {'formula − CatBoost': res['formula'] - res['CatBoost'], 'blend − formula': res['blend 0.75/0.25'] - res['formula'],
            'residual CB − formula': res['formula + residual CB'] - res['formula'], 'oracle − formula': res['oracle (true p*)'] - res['formula'],
            'CatBoost trees': cb['rounds'], 'residual trees': bi + 1}
    return res

truth: mean p* 0.1117 (real base rate 0.1117); oracle AUC on the real labels 0.7242 (in-sample)


## Real data vs 5 simulations (seed-0 folds)
The oracle on real data is not available (the true p is unknown), so that column is blank for the real row.

In [2]:
rows = {'real data': pipeline(y, 'sim_real')}
rows['real data']['oracle (true p*)'] = np.nan; rows['real data']['oracle − formula'] = np.nan
rng = np.random.default_rng(2026)
for i in range(5):
    ys = (rng.random(len(y)) < p_star).astype(float)
    rows[f'simulation {i + 1}'] = pipeline(ys, f'sim{i + 1}')
    print(f'simulation {i + 1} done: formula {rows[f"simulation {i + 1}"]["formula"]:.4f}, CatBoost {rows[f"simulation {i + 1}"]["CatBoost"]:.4f}', flush=True)
T = pd.DataFrame(rows).T
sims = T.drop('real data')
T.loc['simulation mean'] = sims.mean(); T.loc['simulation sd'] = sims.std()
cols = ['formula', 'CatBoost', 'formula − CatBoost', 'blend − formula', 'residual CB − formula', 'oracle (true p*)', 'oracle − formula', 'CatBoost trees', 'residual trees']
T[cols].style.format({c: '{:+.4f}' if '−' in c else '{:.4f}' for c in cols} | {'CatBoost trees': '{:.0f}', 'residual trees': '{:.0f}'}, na_rep='')

simulation 1 done: formula 0.7211, CatBoost 0.7102


simulation 2 done: formula 0.7270, CatBoost 0.7147


simulation 3 done: formula 0.7200, CatBoost 0.7080


simulation 4 done: formula 0.7164, CatBoost 0.7061


simulation 5 done: formula 0.7176, CatBoost 0.7061


,formula,CatBoost,formula − CatBoost,blend − formula,residual CB − formula,oracle (true p*),oracle − formula,CatBoost trees,residual trees
real data,0.7206,0.7145,+0.0061,+0.0007,+0.0006,,,489,587
simulation 1,0.7211,0.7102,+0.0110,-0.0005,+0.0000,0.7234,+0.0023,343,1
simulation 2,0.7270,0.7147,+0.0123,-0.0006,-0.0000,0.7284,+0.0014,395,1
simulation 3,0.7200,0.7080,+0.0120,-0.0004,+0.0000,0.7225,+0.0025,326,6
simulation 4,0.7164,0.7061,+0.0103,-0.0003,+0.0000,0.7184,+0.0020,522,1
simulation 5,0.7176,0.7061,+0.0115,-0.0005,-0.0000,0.7194,+0.0018,341,1
simulation mean,0.7204,0.7090,+0.0114,-0.0005,+0.0000,0.7224,+0.0020,385,2
simulation sd,0.0041,0.0036,+0.0008,+0.0001,+0.0000,0.0039,+0.0004,81,2


In [3]:
# Where does the real data fall within the simulation distribution? (z = (real − sim mean) / sim sd)
z = {c: (T.loc['real data', c] - T.loc['simulation mean', c]) / T.loc['simulation sd', c] for c in ['formula − CatBoost', 'blend − formula', 'residual CB − formula']}
pd.Series(z, name='z of real data vs simulations').round(2).to_frame()

,z of real data vs simulations
formula − CatBoost,-6.70
blend − formula,9.32
residual CB − formula,28.88


**Reading: the real data does NOT behave like "formula + coin flips".**
- **CatBoost is relatively better on real data.** When the formula is the truth, it beats CatBoost by **0.0114 ± 0.0008**; on real data by only **0.0061** (z −6.7). CatBoost finds ~0.005 of real structure that a formula-generated world wouldn't contain.
- **The blend only helps on real data.** With the formula as truth, blending in CatBoost *hurts* (−0.0005 ± 0.0001); on real data it helps (+0.0007; z +9). So the blend gain is mostly CatBoost's extra signal, not noise reduction (correcting my earlier guess).
- **The residual model only finds something on real data.** With the formula as truth, the residual CatBoost stops at 1–6 trees and gains exactly 0. On real data it uses 587 trees and gains +0.0006 (seed-0 folds).
- **The cost of estimating the formula:** with the true structure known, a refit loses ~0.002 AUC to the oracle (0.7224 vs 0.7204), just from estimating 29 coefficients on 30k rows.

## Conclusions
- **The formula is the backbone, not the whole generator.** Its own AUC on real data matches the simulations (0.7206 vs 0.7204), so its terms are right. But real data contains **additional structure the formula lacks**: CatBoost recovers part of it, which explains its 0.005 relative advantage over the simulations, the blend gain and the residual gain.
- **Size:** CatBoost captures at most ~0.0005–0.0007 of it on top of the formula. The full missing piece could be larger, since CatBoost learns it inefficiently.
- **Where to look:** the residual models (`12`, `13`) consistently point at columns the formula dropped: driver safety and accident-free years, driver commute, Census commute time and owner share. Those were only offered to the formula as **linear** terms (in `12`/`13` step A). The generator may use them with **thresholds or other non-linear shapes**, like every other rule we found.
- **Next:** offer the dropped columns to the formula in step / band / spline forms (per era), guided by the residual CatBoost's SHAP dependence plots.